In [ ]:
import pandas

financiera = pandas.read_csv("financiera.csv")

financiera

,folio,amortizacionId,numero,concepto,plan,plazos,precio,interes,anticipo,subtotal,...,pagoMontoFaltante,pagoAnteriorId,pagoSiguienteId,vendedorRecibeFolio,sucursalRecibeFolio,vendedorOrigenFolio,sucursalOrigenFolio,liquidado,amortizacionAnteriorId,amortizacionSiguienteId
0,772,anticipo,0,Anticipo,nuevo,26,3999.0,190.0,799.0,6080.0,...,0.00,NaN,pago-1,33.0,6.0,12.0,14.0,0,NaN,pago-1
1,772,pago-1,1,Pago 1 de 26,nuevo,26,3999.0,190.0,799.0,6080.0,...,233.85,anticipo,pago-2,NaN,NaN,NaN,NaN,0,anticipo,pago-2
2,772,pago-2,2,Pago 2 de 26,nuevo,26,3999.0,190.0,799.0,6080.0,...,233.85,pago-1,pago-3,NaN,NaN,NaN,NaN,0,pago-1,pago-3
3,772,pago-3,3,Pago 3 de 26,nuevo,26,3999.0,190.0,799.0,6080.0,...,233.85,pago-2,pago-4,NaN,NaN,NaN,NaN,0,pago-2,pago-4
4,772,pago-4,4,Pago 4 de 26,nuevo,26,3999.0,190.0,799.0,6080.0,...,233.85,pago-3,pago-5,NaN,NaN,NaN,NaN,0,pago-3,pago-5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13916,9,pago-14,14,Pago 14 de 18,nuevo,18,3999.0,170.0,799.0,5440.0,...,0.00,pago-13,pago-15,18.0,11.0,14.0,11.0,0,pago-13,pago-15
13917,9,pago-15,15,Pago 15 de 18,nuevo,18,3999.0,170.0,799.0,5440.0,...,0.00,pago-14,pago-16,14.0,11.0,14.0,11.0,0,pago-14,pago-16
13918,9,pago-16,16,Pago 16 de 18,nuevo,18,3999.0,170.0,799.0,5440.0,...,0.00,pago-15,pago-17,18.0,11.0,14.0,11.0,0,pago-15,pago-17
13919,9,pago-17,17,Pago 17 de 18,nuevo,18,3999.0,170.0,799.0,5440.0,...,0.00,pago-16,pago-18,18.0,11.0,14.0,11.0,0,pago-16,pago-18


In [2]:
ventas = financiera.groupby("folio").agg({
    "plan": "first",
    "plazos": "first",
    "precio": "first",
    "anticipo": "first",
    "interes": "first",
    "subtotal": "first",
    "total": "first",
    "semanal": "first",
    "vendedorOrigenFolio": "first",
    "sucursalOrigenFolio": "first",
})

ventas

,plan,plazos,precio,anticipo,interes,subtotal,total,semanal,vendedorOrigenFolio,sucursalOrigenFolio
folio,,,,,,,,,,
9,nuevo,18,3999.0,799.0,170.0,5440.00,6239.00,302.22,14.0,11.0
14,renovacion,35,10999.0,0.0,165.0,18148.35,18148.35,518.52,15.0,12.0
17,nuevo,35,10999.0,2499.0,210.0,17850.08,20349.08,510.00,25.0,12.0
18,renovacion,43,8799.0,0.0,170.0,14958.30,14958.30,347.87,15.0,12.0
23,nuevo,15,3999.0,799.0,150.0,4800.00,5599.00,320.00,10100.0,6.0
...,...,...,...,...,...,...,...,...,...,...
764,renovacion,43,8999.0,0.0,100.0,8999.00,8999.00,209.28,10001.0,3.0
766,renovacion,15,9499.0,0.0,150.0,14248.50,14248.50,949.90,31.0,13.0
767,renovacion,26,6699.0,749.5,160.0,9519.20,10268.70,366.12,10.0,5.0


In [ ]:
! python -m pip install pyodbc

In [8]:
import pyodbc

driver = "ODBC Driver 18 for SQL Server"
server = "DESKTOP-IUGR5BT\\SQLEXPRESS"
database = "Financiera2"

conexion = pyodbc.connect(
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

conexion

In [9]:
query = (
    """
        INSERT INTO dbo.ventas (
            folio,
            tipo,
            plazos,
            precio,
            anticipo,
            interes,
            subtotal,
            total,
            semanal,
            vendedorFolio,
            sucursalFolio
        ) VALUES (
            ?,
            ?,
            ?,
            ?,
            ?,
            ?,
            ?,
            ?,
            ?,
            ?,
            ?
        );
    """
)

In [14]:
values = [
    (
        int(folio),
        plan,
        int(plazos),
        float(precio),
        float(anticipo),
        float(interes),
        float(subtotal),
        float(total),
        float(semanal),
        int(vendedorFolio),
        int(sucursalFolio),
    ) for (
        folio,
        plan,
        plazos,
        precio,
        anticipo,
        interes,
        subtotal,
        total,
        semanal,
        vendedorFolio,
        sucursalFolio
    ) in ventas.itertuples()
]

In [15]:
cursor = conexion.cursor()

cursor.fast_executemany = True
cursor.executemany(query, values)
cursor.commit()

cursor.close()

In [16]:
conexion.close()